# F1_t: F1과 같은 시간 가드 융합, 단 v20의 한글표만 72B 표로 바꾼 v20t 사용
**GPU 런타임(L4)**으로 위에서부터 실행하세요. 마지막에 `F1_t: PASS`가 나오면 `combo_kit/out/F1_t.zip`이 생깁니다. `FAIL`이면 올리지 마세요.
`T_c`는 F1 때와 같은 값(18.8)을 그대로 둡니다. v20t는 표 파일 하나만 다르므로 실행 시간은 F1과 같습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/combo_kit"
os.chdir(D)
!ls -la build_fuse_safe.py make_v20t.py catalog_ko_72b.json v20.zip 112_submit.zip check_combo.py
!nvidia-smi --query-gpu=name --format=csv
!pip -q install lightgbm==4.6.0 catboost==1.2.8 rapidfuzz==3.14.3 transformers==4.57.6 accelerate==1.9.0 scikit-learn==1.5.2


In [ ]:
T_c = 18.8   # <- 115(C_vote)의 서버 실행 시간(분)
BUD = int(min(24, 28.5 - max(0.0, T_c - 11.8)))
BUD = max(BUD, 18)
print("v20 시간 예산(분):", BUD)
W = "/content/combo"; os.makedirs(W, exist_ok=True); os.makedirs(f"{D}/out", exist_ok=True)


In [ ]:
# v20t 만들기(표 파일만 교체) → 조립 (약 20분)
zp = f"{W}/F1_t.zip"
for f in (zp, f"{W}/v20t.zip"):
    if os.path.exists(f): os.remove(f)
!python make_v20t.py {W}/v20t.zip v20.zip catalog_ko_72b.json
!unzip -p {W}/v20t.zip model/catalog_ko.json | head -c 200; echo
!python build_fuse_safe.py {zp} {W}/v20t.zip 112_submit.zip 4 {BUD}
os.remove(f"{W}/v20t.zip")


In [ ]:
# 점검 (약 20분): 3,000행 실행 + 영문 불변 + 600행 행 독립성. 통과할 때만 out/F1_t.zip으로 복사
c = subprocess.run([sys.executable, "check_combo.py", zp, f"{W}/work_F1_t"], capture_output=True, text=True)
log = c.stdout + "\n" + c.stderr
open(f"{W}/F1_t_check.log", "w").write(log)
print("\n".join(l for l in log.splitlines() if l.startswith(("[1]", "[2]", "[3]", "zip size", "router start", "OK", "PROBLEM", "WARNING ko", "WARNING en")) or "predictions to output" in l))
bad = [k for k in ("WARNING ko", "WARNING en", "still running", "did not start", "en failed") if k in log]
last = (c.stdout.strip().splitlines() or [""])[-1]
ok = c.returncode == 0 and last == "OK" and not bad and log.count("fell back 0,") == 2 and os.path.getsize(zp) < 10e9
if ok:
    shutil.copy(zp, f"{D}/out/F1_t.zip"); print("F1_t: PASS ->", f"{D}/out/F1_t.zip", round(os.path.getsize(f"{D}/out/F1_t.zip")/1e9, 2), "GB")
else:
    print("F1_t: FAIL - 올리지 마세요. 문제:", bad or last, " 전체 로그:", f"{W}/F1_t_check.log")
